# Manual SARIMA Model

## Purpose
Fits a manually specified **SARIMA(1,0,0)(1,1,0,12)** model using walk-forward validation as an initial proof-of-concept. Unlike the ARIMA notebooks, no manual seasonal differencing is required — `SARIMAX` accepts the raw series and handles the seasonal difference internally via `D=1`.

This notebook precedes the automated grid search and serves to validate the walk-forward pipeline before scaling it across a large parameter space.

## Inputs
- `data/dataset.csv` — Training dataset (93 monthly observations)

## Outputs
- The walk-forward predictions and RMSE.

In [1]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
from math import sqrt

## Load Training Data

In [2]:
series = pd.read_csv('data/dataset.csv', index_col=0, parse_dates=True).iloc[:, 0]
series.head()

Month
1964-01-01    2815
1964-02-01    2672
1964-03-01    2755
1964-04-01    2721
1964-05-01    2946
Name: Sales, dtype: int64

## Train / Test Split

50/50 split, consistent with the persistence baseline notebook, enabling a direct RMSE comparison.

In [3]:
# prepare data
X = series.values
X = X.astype('float64')
train_size = int(len(X) * 0.50)
train, test = X[0:train_size], X[train_size:]

## Walk-Forward Validation with SARIMA(1,0,0)(1,1,0,12)

At each step:
1. Fit a **SARIMA(1,0,0)(1,1,0,12)** model directly on the raw history window. `SARIMAX` applies the seasonal difference (`D=1`) internally, so no manual differencing is needed here.
2. Forecast one step ahead on the original scale.
3. Append the true observation to history before moving to the next step.

The model is re-fit from scratch at every step to avoid look-ahead bias.

In [4]:
# walk-forward validation
# SARIMA(1,0,0)(1,1,0,12): AR(1) non-seasonal, SAR(1) + seasonal difference
# SARIMAX operates on the raw series — no manual differencing required
history = [x for x in train]
predictions = list()
order = (1, 0, 0)
seasonal_order = (1, 1, 0, 12)
for i in range(len(test)):
    model = SARIMAX(history, order=order, seasonal_order=seasonal_order,
                    enforce_stationarity=False, enforce_invertibility=False)
    model_fit = model.fit(disp=False)
    yhat = model_fit.forecast()[0]
    predictions.append(yhat)
    obs = test[i]
    history.append(obs)
    print('>Predicted=%.3f, Expected=%.3f' % (yhat, obs))

>Predicted=7979.843, Expected=8314.000
>Predicted=9811.093, Expected=10651.000
>Predicted=7049.434, Expected=3633.000
>Predicted=2686.397, Expected=4292.000
>Predicted=3475.879, Expected=4154.000
>Predicted=3919.828, Expected=4121.000
>Predicted=4301.402, Expected=4647.000
>Predicted=4492.053, Expected=4753.000
>Predicted=3766.813, Expected=3965.000
>Predicted=1975.014, Expected=1723.000
>Predicted=3976.710, Expected=5048.000
>Predicted=5837.307, Expected=6922.000
>Predicted=8933.212, Expected=9858.000
>Predicted=11139.239, Expected=11331.000
>Predicted=5966.241, Expected=4016.000
>Predicted=3870.089, Expected=3957.000
>Predicted=4033.537, Expected=4510.000
>Predicted=4469.751, Expected=4276.000
>Predicted=4633.686, Expected=4968.000
>Predicted=4863.762, Expected=4677.000
>Predicted=3874.762, Expected=3523.000
>Predicted=1530.378, Expected=1821.000
>Predicted=5020.116, Expected=5222.000
>Predicted=6639.690, Expected=6872.000
>Predicted=9619.020, Expected=10803.000
>Predicted=11733.130,

## Report RMSE

Compare this RMSE against the persistence baseline. An improvement confirms that SARIMA structure adds predictive value over the naïve model and justifies proceeding to the grid search.

In [5]:
# report performance
rmse = sqrt(mean_squared_error(test, predictions))
print('RMSE: %.3f' % rmse)

RMSE: 1036.167
